# 00 — Comprendre le panel télécom, puis analyser les KPI de churn

**Objectif :** comprendre la composition du jeu de données avant de chercher des associations avec le churn. Le notebook 01 est la référence de lecture, normalisation et gestion des doublons ; le 02 entraîne les modèles et le 03 approfondit leur évaluation et leur explicabilité.

**Parcours :** structure et qualité → clients et présence mensuelle → trajectoires et contrats → distributions et relations entre colonnes → dix KPI de churn → synthèse.

Une ligne représente un **client observé à une fin de mois**. `churn=1` désigne une résiliation dans les 30 jours suivants, selon l'énoncé. Un client peut donc contribuer plusieurs lignes, mais une clé `(customer_id, snapshot_date)` doit être unique. Les proportions par snapshot et par client ont des dénominateurs différents.


## 0 — Environnement et conventions

Même import de configuration que le notebook 01. Les petits helpers ci-dessous servent uniquement à afficher les analyses dans ce notebook.

In [ ]:
from pathlib import Path
import sys
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display, Markdown

if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

SEED = 42
COLORS = {"blue": "#2563EB", "red": "#E11D48", "green": "#059669", "dark": "#0F172A"}
CLASS_COLORS = [COLORS["green"], COLORS["red"]]
from io import StringIO
from IPython.display import HTML

FIG_WIDTH = 11  # Largeur de référence en pouces, ajustable.
PANEL_HEIGHT = 3.8
plt.rcParams.update({
    "axes.grid": True, "grid.alpha": .2, "figure.dpi": 110,
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})

def show_figure(fig):
    """Affichage SVG qui s'adapte à la largeur disponible sans écrire de fichier."""
    fig.tight_layout()
    buffer = StringIO()
    fig.savefig(buffer, format="svg", bbox_inches="tight")
    svg = buffer.getvalue()
    svg = svg.replace('<svg ', '<svg style="display:block;width:100%;height:auto;" ', 1)
    display(HTML('<div style="width:100%;max-width:100%;">' + svg + '</div>'))
    plt.close(fig)


def summary(data, groups):
    """Taux par snapshot étiqueté ; comptes clients distincts non additifs entre groupes."""
    return (data.dropna(subset=["churn"]).groupby(groups, dropna=False, observed=True)
            .agg(observations=("churn", "size"), positives=("churn", "sum"),
                 clients_uniques=("customer_id", "nunique"), churn_rate=("churn", "mean"))
            .reset_index().assign(churn_rate=lambda frame: frame.churn_rate * 100))


def rate_plot(stats, group, title, *, horizontal=False):
    """Afficher les taux et le repère global ; les effectifs restent dans le tableau."""
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(stats) * .4) if horizontal else PANEL_HEIGHT))
    labels = stats[group].astype("string").fillna("Non renseigné").tolist()
    if stats.empty:
        ax.text(.5, .5, "Aucun groupe exploitable", ha="center", transform=ax.transAxes)
    elif horizontal:
        bars = ax.barh(labels, stats.churn_rate.to_numpy(), color=COLORS["blue"])
        ax.bar_label(bars, fmt="%.2f %%", padding=3)
        ax.axvline(global_rate, color=COLORS["red"], linestyle="--", label="Prévalence globale")
        ax.set_xlabel("Churn par snapshot (%)")
        ax.invert_yaxis()
    else:
        bars = ax.bar(labels, stats.churn_rate.to_numpy(), color=COLORS["blue"])
        ax.bar_label(bars, fmt="%.2f %%", padding=3)
        ax.axhline(global_rate, color=COLORS["red"], linestyle="--", label="Prévalence globale")
        ax.set_ylabel("Churn par snapshot (%)")
        ax.tick_params(axis="x", labelrotation=25)
    ax.set_title(title)
    ax.margins(x=.15 if horizontal else .05, y=.2)
    if not stats.empty:
        ax.legend()
    show_figure(fig)
    display(stats)

## 1 — Lecture et audit des doublons : même politique que le notebook 01

Les identifiants et départements restent du texte. On normalise les dates, les numériques et les catégories. Les copies exactes sont supprimées ; toutes les lignes d'une clé client × snapshot ambiguë sont exclues, sans arbitrage. Les diagnostics restent visibles. `raw_df` conserve les lignes sources pour comparer avant/après ; cette politique ne corrige pas des conflits métier à la source.

In [ ]:
df = pd.read_csv(settings.DATA_PATH, dtype={"customer_id": "string", "dept": "string"})
raw_df = df.copy()

required = [
    "customer_id", "snapshot_date", "signup_date", "age", "contract",
    "region", "dept", "channel", "monthly_fee", "data_gb",
    "support_calls_3m", "late_payments", "nps", "last_invoice_amount",
    "retention_offer_declined", "churn",
]
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f"Colonnes absentes : {missing_columns}")

df["customer_id"] = df["customer_id"].str.strip()

for column in ["snapshot_date", "signup_date"]:
    df[column] = pd.to_datetime(df[column], errors="coerce")

numeric_columns = [
    "age", "monthly_fee", "data_gb", "support_calls_3m",
    "late_payments", "nps", "last_invoice_amount", "churn",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["contract", "region", "dept", "channel"]:
    df[column] = df[column].astype("string").str.strip().str.strip("\"'").str.lower()

if df["snapshot_date"].isna().any():
    raise ValueError("Des snapshot_date sont invalides.")
if df["customer_id"].isna().any() or df["customer_id"].eq("").any():
    raise ValueError("Des identifiants client sont manquants.")
# Audit et résolution prudente des doublons client × mois.
# Les copies parfaitement identiques sont dédupliquées.
# Les clés présentant des valeurs divergentes sont exclues (jamais arbitrées au hasard).
KEY = ["customer_id", "snapshot_date"]

raw_count = len(df)
df = df.drop_duplicates().copy()
exact_removed = raw_count - len(df)

ambiguous_mask = df.duplicated(KEY, keep=False)
ambiguous_rows = df.loc[ambiguous_mask].copy()
ambiguous_keys = ambiguous_rows[KEY].drop_duplicates()

print(f"Lignes chargées : {raw_count:,}")
print(f"Copies strictement identiques supprimées : {exact_removed:,}")
print(f"Clés client × snapshot ambiguës : {len(ambiguous_keys):,}")

if not ambiguous_rows.empty:
    # Diagnostic de conflits de label, sans modifier la cible.
    label_conflicts = (
        ambiguous_rows.groupby(KEY, dropna=False)["churn"]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )
    print(f"Clés avec labels contradictoires ou manquants : {label_conflicts:,}")
    print(f"Lignes ambiguës exclues : {len(ambiguous_rows):,}")
    display(ambiguous_rows.sort_values(KEY).head(12))

    # Une clé ambiguë entière est exclue pour éviter une sélection arbitraire.
    df = df.loc[~ambiguous_mask].copy()

if df.empty:
    raise ValueError("Aucune observation non ambiguë après audit des doublons.")

assert not df.duplicated(KEY).any()
print(f"Lignes conservées après audit : {len(df):,}")
if not df["churn"].dropna().isin([0, 1]).all():
    raise ValueError("churn doit être binaire (0 ou 1).")

labeled = df.dropna(subset=["churn"]).copy()
if labeled.empty:
    raise ValueError("Aucune observation avec un label churn valide.")
labeled["churn"] = labeled["churn"].astype(int)

print("Snapshots étiquetés :", len(labeled))
print("Clients distincts :", labeled["customer_id"].nunique())
print("Prévalence churn :", f"{labeled['churn'].mean():.2%}")
print("Période :", labeled["snapshot_date"].min().date(), "→", labeled["snapshot_date"].max().date())

display(labeled.head(3))

## 2 — Quelles informations contient chaque colonne ?

**Question :** Quels sont les types, les rôles, les cardinalités et les manquants ?

**Méthode :** Comparer le CSV brut aux colonnes normalisées, puis afficher un dictionnaire de lecture et les taux de valeurs manquantes.

**À lire :** Une forte cardinalité est attendue pour un identifiant, pas nécessairement pour une catégorie métier. Une absence de valeur ne signifie pas zéro.

In [ ]:
roles = {
    "customer_id": "Identifiant client ; répétition mensuelle attendue",
    "snapshot_date": "Date de référence de l'observation",
    "signup_date": "Début de relation ; source d'ancienneté",
    "age": "Démographie ; EDA uniquement, exclue du modèle 02",
    "contract": "Type d'engagement", "region": "Région", "dept": "Département (texte)",
    "channel": "Canal", "monthly_fee": "Mensualité (€)", "data_gb": "Consommation (Go)",
    "support_calls_3m": "Appels au support sur 3 mois", "late_payments": "Retards ; fenêtre à confirmer",
    "nps": "Satisfaction sur l'échelle 0–10", "last_invoice_amount": "Facture ; temporalité à vérifier",
    "retention_offer_declined": "Intervention commerciale ; temporalité à vérifier, exclue du modèle",
    "churn": "Cible binaire à 30 jours",
}
column_profile = pd.DataFrame({
    "colonne": raw_df.columns,
    "rôle": [roles.get(c, "À documenter") for c in raw_df.columns],
    "type_brut": [str(raw_df[c].dtype) for c in raw_df.columns],
    "type_normalisé": [str(df[c].dtype) for c in raw_df.columns],
    "valeurs_distinctes": [df[c].nunique(dropna=True) for c in raw_df.columns],
    "manquants_bruts": [raw_df[c].isna().sum() for c in raw_df.columns],
    "manquants_retenus": [df[c].isna().sum() for c in raw_df.columns],
    "manquants_pct": [df[c].isna().mean() * 100 for c in raw_df.columns],
})
display(column_profile)
print(f"Brut : {raw_df.shape} ; après audit : {df.shape}")
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(column_profile) * .3)))
ordered = column_profile.sort_values("manquants_pct", ascending=False)
ax.barh(ordered.colonne, ordered.manquants_pct, color=COLORS["blue"])
ax.invert_yaxis()
ax.set(title="Valeurs manquantes après normalisation et audit", xlabel="Part des snapshots (%)")
show_figure(fig)

most_missing = column_profile.sort_values("manquants_pct", ascending=False).iloc[0]
display(Markdown(f"**Constat calculé :** `{most_missing.colonne}` est la colonne la plus manquante ({most_missing.manquants_pct:.1f} % des snapshots retenus). Les taux des sections suivantes doivent être lus avec leur nombre de valeurs disponibles, particulièrement pour cette colonne."))


## 3 — Quelles anomalies et variantes de saisie observe-t-on ?

**Question :** Les catégories changent-elles seulement par la casse ou les espaces ? Les numériques respectent-ils leur domaine ?

**Méthode :** Afficher les fréquences brutes et normalisées ; compter les valeurs hors domaine et les conversions devenues manquantes.

**À lire :** Les contrôles séparent qualité de saisie et réalité métier. Aucune imputation automatique. Les valeurs invalides sont neutralisées uniquement dans la copie EDA utilisée pour les KPI, selon les domaines du feature engineering du 01.

In [ ]:
for column in ["contract", "region", "dept", "channel"]:
    before = raw_df[column].value_counts(dropna=False).rename_axis("modalité_brute").reset_index(name="snapshots_bruts")
    after = df[column].value_counts(dropna=False).rename_axis("modalité_normalisée").reset_index(name="snapshots_retenus")
    print(column, "— avant / après")
    display(before, after)

quality_rows = []
for column in numeric_columns:
    converted = pd.to_numeric(raw_df[column], errors="coerce")
    quality_rows.append({"colonne": column,
                         "échecs_conversion": int((raw_df[column].notna() & converted.isna()).sum()),
                         "valeurs_négatives": int(df[column].lt(0).sum()),
                         "non_finies": int((df[column].notna() & ~np.isfinite(df[column])).sum())})
display(pd.DataFrame(quality_rows))
display(pd.Series({
    "signup_date manquante ou invalide": int(df.signup_date.isna().sum()),
    "signup_date après snapshot": int((df.signup_date > df.snapshot_date).sum()),
    "snapshot hors fin de mois": int((~df.snapshot_date.dt.is_month_end).sum()),
    "NPS hors [0,10]": int((df.nps.notna() & ~df.nps.between(0, 10)).sum()),
    "âge hors [0,120] (contrôle descriptif)": int((df.age.notna() & ~df.age.between(0, 120)).sum()),
    "offre refusée hors {0,1}": int((df.retention_offer_declined.notna() & ~df.retention_offer_declined.isin([0, 1])).sum()),
}, name="anomalies").to_frame())
eda = df.copy()
for column in ["monthly_fee", "last_invoice_amount", "data_gb", "support_calls_3m", "late_payments"]:
    eda[column] = eda[column].where(eda[column].ge(0) & np.isfinite(eda[column]))
eda["nps"] = eda.nps.where(eda.nps.between(0, 10))
eda["tenure_months"] = (eda.snapshot_date - eda.signup_date).dt.days / 30.4375
eda["tenure_months"] = eda.tenure_months.where(eda.tenure_months.ge(0))
global_rate = labeled.churn.mean() * 100

display(Markdown(f"**Constat calculé :** le contrat passe de {raw_df.contract.nunique()} modalités brutes à {df.contract.nunique()} après normalisation et audit. Une comparaison non normalisée aurait séparé artificiellement certaines formes de saisie. Les lignes initiales et normalisées restent distinctes en mémoire."))


## 4 — Combien de clients et combien de lignes par client ?

**Question :** Un identifiant est-il unique ? Quelle proportion des clients revient plusieurs mois ?

**Méthode :** Compter les snapshots par client, vérifier la clé client × date, comparer le poids des clients vus une ou plusieurs fois.

**À lire :** Les clients ne sont pas des lignes indépendantes. Un client observé 12 fois pèse 12 fois dans les distributions par snapshot.

In [ ]:
client_counts = df.groupby("customer_id").size().rename("snapshots")
client_structure = pd.Series({
    "lignes brutes": len(raw_df), "lignes retenues": len(df),
    "clients distincts": len(client_counts), "clients vus une fois": int(client_counts.eq(1).sum()),
    "clients vus plusieurs fois": int(client_counts.gt(1).sum()),
    "clés client × snapshot répétées": int(df.duplicated(KEY).sum()),
    "snapshots sans cible": int(df.churn.isna().sum()),
}, name="effectif")
display(client_structure.to_frame())
count_distribution = client_counts.value_counts().sort_index().rename_axis("snapshots_par_client").to_frame("clients")
count_distribution["part_clients_pct"] = count_distribution.clients / len(client_counts) * 100
display(count_distribution)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
bars = ax.bar(count_distribution.index.astype(str), count_distribution.clients, color=COLORS["blue"])
ax.bar_label(bars, padding=3)
ax.set(title="Distribution du nombre de snapshots par client", xlabel="Snapshots observés par client", ylabel="Clients distincts")
show_figure(fig)
print(f"{client_counts.gt(1).mean():.1%} des clients ont plusieurs observations ; médiane : {client_counts.median():.0f} snapshots.")

## 5 — Comment le panel se répartit-il entre les mois ?

**Question :** Combien de lignes, clients, nouveaux entrants et clients déjà présents voit-on chaque mois ?

**Méthode :** Compter les clients distincts et suivre les ensembles d’identifiants par rapport au mois précédent, en complétant les mois absents.

**À lire :** « Première apparition » est une entrée dans le fichier, pas nécessairement une souscription. Une sortie du panel ne prouve pas une résiliation.

In [ ]:
panel = df.assign(month=df.snapshot_date.dt.to_period("M"))
all_months = pd.period_range(panel.month.min(), panel.month.max(), freq="M")
first_month = panel.groupby("customer_id").month.min()
monthly_rows = []
previous_clients = set()
for position, month in enumerate(all_months):
    part = panel.loc[panel.month.eq(month)]
    current_clients = set(part.customer_id)
    monthly_rows.append({"month": str(month), "snapshots": len(part), "clients": len(current_clients),
                         "première_apparition": int(first_month.eq(month).sum()),
                         "déjà_présents_mois_précédent": len(current_clients & previous_clients),
                         "entrées_ou_retours": len(current_clients - previous_clients) if position else np.nan,
                         "absents_depuis_mois_précédent": len(previous_clients - current_clients) if position else np.nan,
                         "cibles_renseignées": int(part.churn.notna().sum()),
                         "churns": int(part.churn.sum())})
    previous_clients = current_clients
monthly_panel = pd.DataFrame(monthly_rows)
display(monthly_panel)
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.7), sharex=True)
axes[0].plot(monthly_panel.month, monthly_panel.clients, "o-", color=COLORS["blue"], label="Clients distincts")
axes[0].plot(monthly_panel.month, monthly_panel.snapshots, "x--", color=COLORS["dark"], label="Snapshots")
axes[0].set(title="Volumes mensuels du panel", ylabel="Effectif")
axes[0].legend()
axes[1].bar(monthly_panel.month, monthly_panel.première_apparition, color=COLORS["green"])
axes[1].set(title="Premières apparitions dans le fichier", ylabel="Clients", xlabel="Mois du snapshot")
axes[1].tick_params(axis="x", labelrotation=35)
show_figure(fig)

display(Markdown(f"**Constat calculé :** le nombre de clients mensuels varie de {monthly_panel.clients.min():,} à {monthly_panel.clients.max():,}. La première période observée contient déjà {int(monthly_panel.première_apparition.iloc[0]):,} clients : il s'agit d'un stock initial du fichier, pas de nouvelles souscriptions de ce mois. Comparer les taux mensuels impose de tenir compte de cette composition changeante."))


## 6 — Les mêmes clients sont-ils présents tous les mois ?

**Question :** Quelles cohortes restent observées ? Existe-t-il des interruptions de suivi ?

**Méthode :** Construire la présence client × mois, puis une matrice d’effectifs par mois de première apparition. Repérer les mois manquants entre première et dernière présence.

**À lire :** La carte de cohortes compte des clients observés ; ce n’est pas une courbe de survie. Les premières présences en janvier peuvent correspondre à des clients anciens.

In [ ]:
presence = pd.crosstab(panel.customer_id, panel.month).reindex(columns=all_months, fill_value=0).gt(0)
first_position = presence.to_numpy().argmax(axis=1)
last_position = len(all_months) - 1 - presence.to_numpy()[:, ::-1].argmax(axis=1)
internal_gaps = last_position - first_position + 1 - presence.sum(axis=1).to_numpy()
print("Clients avec au moins un mois absent entre deux observations :", int((internal_gaps > 0).sum()))
cohort_panel = panel.assign(cohort=panel.customer_id.map(first_month))
cohort_counts = pd.crosstab(cohort_panel.cohort, cohort_panel.month).reindex(columns=all_months, fill_value=0)
cohort_counts = cohort_counts.reindex(index=sorted(cohort_counts.index))
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(cohort_counts) * .45)))
im = ax.imshow(cohort_counts.to_numpy(), aspect="auto", cmap="Blues")
ax.set_xticks(range(len(all_months)), [str(m) for m in all_months], rotation=40)
ax.set_yticks(range(len(cohort_counts)), cohort_counts.index.astype(str))
ax.set(title="Clients observés par cohorte et mois", xlabel="Mois observé", ylabel="Mois de première apparition")
for i in range(len(cohort_counts)):
    for j in range(len(all_months)):
        value = cohort_counts.iloc[i, j]
        if value:
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > cohort_counts.to_numpy().max() / 2 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Clients observés")
show_figure(fig)
display(cohort_counts)

display(Markdown(f"**Constat calculé :** {int((internal_gaps > 0).sum()):,} clients ont un trou entre deux présences ; {int(presence.all(axis=1).sum()):,} sont présents pendant tous les mois du fichier. Une courte trajectoire peut venir d'une arrivée tardive ou d'une sortie ; elle ne suffit pas, seule, à identifier sa cause."))


## 7 — Churn et non-churn : quelle unité compte-t-on ?

**Question :** Comment les clients se répartissent-ils selon leur churn observé ? Y a-t-il des churns multiples ou des lignes après churn ?

**Méthode :** Comparer les classes par snapshot à la présence d’au moins un churn par client, puis auditer les trajectoires triées.

**À lire :** « Aucun churn observé » n’est pas « ne résiliera jamais ». La durée de suivi varie. La part de clients ayant un churn sur la période n’est pas le risque mensuel.

In [ ]:
client_labels = df.groupby("customer_id").churn.agg(observed_labels="count", positive_snapshots="sum")
client_status = pd.Series("Aucun churn observé", index=client_labels.index)
client_status.loc[client_labels.positive_snapshots.gt(0)] = "Au moins un churn observé"
client_status.loc[client_labels.observed_labels.eq(0)] = "Aucun label disponible"
client_status_counts = client_status.value_counts()
client_status_table = client_status_counts.rename_axis("statut_client").to_frame("clients")
client_status_table["part_pct"] = client_status_table.clients / len(client_status) * 100
display(client_status_table)
ordered_panel = df.sort_values(KEY).copy()
ordered_panel["prior_churn"] = ordered_panel.groupby("customer_id").churn.transform(
    lambda values: values.fillna(0).shift(fill_value=0).cummax())
post_churn = ordered_panel.loc[ordered_panel.prior_churn.eq(1)]
trajectory_audit = pd.Series({
    "clients avec plusieurs snapshots positifs": int(client_labels.positive_snapshots.gt(1).sum()),
    "snapshots observés après un churn positif": len(post_churn),
    "clients avec churn observé": int(client_labels.positive_snapshots.gt(0).sum()),
}, name="effectif")
display(trajectory_audit.to_frame())
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.5))
snapshot_counts = labeled.churn.value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(["Non-churn", "Churn"], snapshot_counts.to_numpy(), color=CLASS_COLORS)
axes[0].set(title=f"Par snapshot étiqueté — prévalence {global_rate:.2f} %", ylabel="Snapshots")
axes[1].barh(client_status_counts.index, client_status_counts.to_numpy(), color=COLORS["blue"])
axes[1].set(title="Par client — statut sur toute la période observée", xlabel="Clients distincts")
show_figure(fig)
print(f"Part des clients avec au moins un churn observé : {client_labels.positive_snapshots.gt(0).mean():.2%}")

display(Markdown(f"**Constat calculé :** {int(client_labels.positive_snapshots.gt(0).sum()):,} clients ont au moins un churn observé, contre {int(client_labels.positive_snapshots.eq(0).sum()):,} sans churn observé. L'audit trouve {int(client_labels.positive_snapshots.gt(1).sum()):,} clients avec plusieurs labels positifs et {len(post_churn):,} lignes après churn. C'est ce contrôle qui permet de discuter le nombre d'événements, plutôt que de supposer l'identifiant unique."))


## 8 — Contrats : clients, snapshots et changements

**Question :** La répartition des contrats est-elle la même selon l’unité ? Un client change-t-il de contrat ?

**Méthode :** Comparer les snapshots et le premier contrat observé de chaque client ; croiser contrat et mois, puis compter les changements au sein d’un client.

**À lire :** Un client n’est compté qu’une fois dans la distribution de son premier contrat. Les clients distincts par contrat, eux, peuvent appartenir à plusieurs contrats sur l’année.

In [ ]:
ordered_clients = df.sort_values(KEY)
first_contract = ordered_clients.drop_duplicates("customer_id", keep="first").contract.fillna("Non renseigné")
contract_distribution = pd.concat([
    df.contract.fillna("Non renseigné").value_counts().rename("snapshots"),
    first_contract.value_counts().rename("clients_premier_contrat"),
    df.groupby("contract", dropna=False).customer_id.nunique().rename("clients_distincts_dans_ce_contrat"),
], axis=1).fillna(0)
contract_distribution["part_snapshots_pct"] = contract_distribution.snapshots / len(df) * 100
contract_distribution["part_clients_premier_contrat_pct"] = contract_distribution.clients_premier_contrat / len(first_contract) * 100
display(contract_distribution)
contract_changes = df.groupby("customer_id").contract.nunique().gt(1)
print("Clients avec plusieurs contrats observés :", int(contract_changes.sum()))
contract_by_month = pd.crosstab(panel.month, panel.contract.fillna("Non renseigné")).reindex(all_months, fill_value=0)
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.7))
contract_distribution[["part_snapshots_pct", "part_clients_premier_contrat_pct"]].plot.bar(ax=axes[0], color=[COLORS["blue"], COLORS["green"]], rot=0)
axes[0].set(title="Composition des contrats selon l'unité", ylabel="Part (%)", xlabel="Contrat")
axes[0].legend(["Snapshots", "Clients : premier contrat observé"])
contract_share = contract_by_month.div(contract_by_month.sum(axis=1).replace(0, np.nan), axis=0) * 100
contract_share.index = contract_share.index.astype(str)
contract_share.plot.bar(stacked=True, ax=axes[1], rot=35)
axes[1].set(title="Composition mensuelle par contrat", ylabel="Part des snapshots (%)", xlabel="Mois")
axes[1].legend(title="Contrat")
show_figure(fig)
display(contract_by_month)

largest_contract = contract_distribution.snapshots.idxmax()
display(Markdown(f"**Constat calculé :** `{largest_contract}` représente {contract_distribution.loc[largest_contract, 'part_snapshots_pct']:.1f} % des snapshots, contre {contract_distribution.loc[largest_contract, 'part_clients_premier_contrat_pct']:.1f} % des premiers contrats clients. {int(contract_changes.sum()):,} clients changent de modalité de contrat observée. Une différence entre les deux parts peut donc refléter les durées de suivi autant que la composition initiale."))


## 9 — Quelles colonnes varient chez un même client ?

**Question :** Quelles informations sont stables et lesquelles évoluent au fil des snapshots ?

**Méthode :** Calculer le nombre de modalités non manquantes distinctes par client pour chaque colonne, hors identifiant et date de snapshot.

**À lire :** Une variation peut être réelle ou une anomalie. Les clients vus une seule fois ne permettent pas d’observer un changement. Une valeur manquante n’est pas considérée comme une nouvelle modalité.

In [ ]:
repeat_clients = client_counts.index[client_counts.gt(1)]
variation_rows = []
for column in [c for c in df.columns if c not in KEY]:
    distinct_per_client = df.loc[df.customer_id.isin(repeat_clients)].groupby("customer_id")[column].nunique(dropna=True)
    eligible = distinct_per_client.ge(2) | distinct_per_client.eq(1)
    variation_rows.append({"colonne": column, "clients_répétés": len(repeat_clients),
                           "clients_avec_valeur": int(eligible.sum()),
                           "clients_avec_plusieurs_valeurs": int(distinct_per_client.gt(1).sum()),
                           "part_clients_répétés_pct": distinct_per_client.gt(1).mean() * 100})
variation_table = pd.DataFrame(variation_rows).sort_values("part_clients_répétés_pct", ascending=False)
display(variation_table)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(variation_table) * .3)))
ax.barh(variation_table.colonne, variation_table.part_clients_répétés_pct, color=COLORS["blue"])
ax.invert_yaxis()
ax.set(title="Colonnes variables chez les clients répétés", xlabel="Clients répétés avec plusieurs valeurs (%)")
show_figure(fig)

varying_columns = variation_table.loc[variation_table.clients_avec_plusieurs_valeurs.gt(0), "colonne"].tolist()
stable_columns = variation_table.loc[variation_table.clients_avec_plusieurs_valeurs.eq(0), "colonne"].tolist()
display(Markdown(f"**Lecture du panel :** colonnes avec plusieurs valeurs observées chez au moins un client répété : {', '.join(varying_columns) or 'aucune'}. Colonnes sans variation non manquante observée : {', '.join(stable_columns) or 'aucune'}. Une colonne sans variation observée n'est pas nécessairement immuable ; le suivi et les manquants limitent ce constat."))


## 10 — Région, département et canal : quelle composition ?

**Question :** Les catégories sont-elles équilibrées et quels segments dominent le fichier ?

**Méthode :** Présenter fréquences, parts de snapshots et clients distincts ; limiter le graphique aux 15 modalités les plus fréquentes.

**À lire :** Le tableau conserve toutes les modalités. Les comptes clients par modalité ne sont pas toujours additifs si un client change de catégorie.

In [ ]:
for column in ["region", "dept", "channel"]:
    temp = df.assign(category=df[column].fillna("Non renseigné"))
    counts = temp.groupby("category").agg(snapshots=("customer_id", "size"), clients=("customer_id", "nunique")).sort_values("snapshots", ascending=False)
    counts["part_snapshots_pct"] = counts.snapshots / len(temp) * 100
    print(column)
    display(counts)
    top = counts.head(15)
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(top) * .3)))
    ax.barh(top.index, top.snapshots, color=COLORS["blue"])
    ax.invert_yaxis()
    ax.set(title=f"Composition de {column} — 15 modalités maximum", xlabel="Snapshots")
    show_figure(fig)

## 11 — Distributions des variables numériques

**Question :** Quelles sont les échelles, les asymétries, les valeurs extrêmes et les différences entre médiane et moyenne ?

**Méthode :** Afficher les quantiles et un histogramme par colonne, avec une grille de deux colonnes ajustable.

**À lire :** Les distributions décrivent des snapshots, pas des individus indépendants. L’âge peut être exploré ici mais reste exclu des prédicteurs. Les valeurs manquantes ne sont pas imputées.

In [ ]:
NUMERIC_EDA = ["age", "monthly_fee", "data_gb", "support_calls_3m", "late_payments", "nps", "last_invoice_amount", "tenure_months"]
display(eda[NUMERIC_EDA].describe(percentiles=[.01, .25, .5, .75, .99]).T)
N_COLUMNS = 2  # Passer à 1 si la fenêtre est très étroite.
n_rows = math.ceil(len(NUMERIC_EDA) / N_COLUMNS)
fig, axes = plt.subplots(n_rows, N_COLUMNS, figsize=(FIG_WIDTH, n_rows * 2.6), squeeze=False)
for ax, column in zip(axes.flat, NUMERIC_EDA):
    values = eda[column].dropna()
    ax.hist(values, bins=30, color=COLORS["blue"], alpha=.8)
    if not values.empty:
        ax.axvline(values.median(), color=COLORS["red"], linestyle="--", label=f"Médiane : {values.median():.2f}")
        ax.legend(fontsize=8)
    ax.set(title=f"{column} — manquants : {eda[column].isna().mean():.1%}", xlabel=column, ylabel="Snapshots")
for ax in list(axes.flat)[len(NUMERIC_EDA):]:
    ax.set_visible(False)
show_figure(fig)

## 12 — Relations entre les colonnes, sans se limiter à la cible

**Question :** Quelles variables numériques évoluent ensemble ? Les catégories sont-elles associées entre elles ?

**Méthode :** Corrélations de Spearman par paires avec effectifs disponibles ; V de Cramér descriptif entre catégories ; deux tableaux croisés normalisés par ligne.

**À lire :** La répétition des clients et les manquants affectent ces associations. Le V n’est pas corrigé du biais de cardinalité ; aucun test de significativité n’est interprété. Une corrélation n’est pas une causalité.

In [ ]:
correlation = eda[NUMERIC_EDA].corr(method="spearman", min_periods=30)
available = eda[NUMERIC_EDA].notna().astype(int)
pair_counts = available.T @ available
display(correlation, pair_counts)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 7))
im = ax.imshow(correlation.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC_EDA)), NUMERIC_EDA, rotation=45, ha="right")
ax.set_yticks(range(len(NUMERIC_EDA)), NUMERIC_EDA)
for i in range(len(NUMERIC_EDA)):
    for j in range(len(NUMERIC_EDA)):
        if pd.notna(correlation.iloc[i, j]):
            ax.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Relations numériques — Spearman par snapshot")
fig.colorbar(im, ax=ax, label="Corrélation de rang")
show_figure(fig)

categorical_columns = ["contract", "region", "dept", "channel"]
cramer_rows = []
for i, left in enumerate(categorical_columns):
    for right in categorical_columns[i + 1:]:
        contingency = pd.crosstab(df[left].fillna("Non renseigné"), df[right].fillna("Non renseigné"))
        denominator = min(contingency.shape) - 1
        v = np.sqrt(chi2_contingency(contingency, correction=False)[0] / (contingency.to_numpy().sum() * denominator)) if denominator > 0 else np.nan
        cramer_rows.append({"colonne_1": left, "colonne_2": right, "V_Cramér": v, "snapshots": int(contingency.to_numpy().sum())})
display(pd.DataFrame(cramer_rows).sort_values("V_Cramér", ascending=False))
for right in ["channel", "region"]:
    cross = pd.crosstab(df.contract.fillna("Non renseigné"), df[right].fillna("Non renseigné"))
    share = cross.div(cross.sum(axis=1), axis=0) * 100
    display(cross, share)
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
    share.plot.barh(stacked=True, ax=ax)
    ax.set(title=f"Composition de {right} au sein des contrats", xlabel="Part des snapshots du contrat (%)", ylabel="Contrat")
    ax.legend(title=right, bbox_to_anchor=(1.02, 1), loc="upper left")
    show_figure(fig)

upper_pairs = correlation.where(np.triu(np.ones(correlation.shape, dtype=bool), k=1)).stack()
if not upper_pairs.empty:
    strongest_pair = upper_pairs.abs().idxmax()
    pair_value = upper_pairs.loc[strongest_pair]
    pair_n = pair_counts.loc[strongest_pair[0], strongest_pair[1]]
    display(Markdown(f"**Constat calculé :** l'association numérique de rang la plus forte parmi les paires étudiées est `{strongest_pair[0]}` / `{strongest_pair[1]}` : Spearman = {pair_value:.3f}, sur {int(pair_n):,} snapshots complets pour cette paire. Ce résultat décrit le fichier ; il ne distingue pas effet métier, construction commune et répétition des clients."))


## 13 — Les manquants sont-ils répartis au hasard dans le fichier ?

**Question :** La disponibilité du NPS, du prix ou de l’âge varie-t-elle avec le mois, le contrat ou le churn ?

**Méthode :** Calculer le pourcentage de manquants par groupe et les effectifs de chaque groupe.

**À lire :** Une non-réponse peut être informative, sans que l’on connaisse son mécanisme. L’association avec le churn ne prouve pas que l’absence cause la résiliation.

In [ ]:
missing_columns = [c for c in NUMERIC_EDA if eda[c].isna().any()]
missing_view = eda.assign(month=eda.snapshot_date.dt.to_period("M").astype(str))
for group in ["month", "contract", "churn"]:
    print("Manquants (%) par", group)
    if not missing_columns:
        print("Aucun manquant parmi les colonnes numériques étudiées.")
        continue
    missing_rates = missing_view.groupby(group, dropna=False)[missing_columns].agg(lambda values: values.isna().mean() * 100)
    group_sizes = missing_view.groupby(group, dropna=False).size().rename("snapshots")
    display(missing_rates.join(group_sizes))
    if group == "month":
        fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
        missing_rates.plot(ax=ax, marker="o")
        ax.set(title="Disponibilité des variables au fil des mois", xlabel="Mois", ylabel="Valeurs manquantes (%)")
        ax.tick_params(axis="x", labelrotation=35)
        show_figure(fig)

## 14 — Facturation et intervention commerciale : une cible presque visible ?

**Question :** La facture ou l’offre refusée séparent-elles directement churn et non-churn ?

**Méthode :** Comparer facture/mensualité lorsque les deux sont valides et la mensualité positive ; croiser les règles observées avec la cible. Afficher les cas non évaluables.

**À lire :** Une séparation parfaite suggère une fuite ou un artefact, pas une causalité. Vérifier les dates de disponibilité avant toute utilisation prédictive. Ces diagnostics descriptifs ne valident pas une règle de ciblage.

In [ ]:
invoice_view = eda.dropna(subset=["churn"]).copy()
valid_invoice = invoice_view.last_invoice_amount.notna() & invoice_view.monthly_fee.gt(0)
invoice_view["invoice_ratio"] = invoice_view.last_invoice_amount / invoice_view.monthly_fee.where(valid_invoice)
invoice_view["invoice_relation"] = "Non évaluable"
invoice_view.loc[valid_invoice & invoice_view.invoice_ratio.eq(1), "invoice_relation"] = "Facture = mensualité"
invoice_view.loc[valid_invoice & invoice_view.invoice_ratio.lt(1), "invoice_relation"] = "Facture < mensualité"
invoice_view.loc[valid_invoice & invoice_view.invoice_ratio.gt(1), "invoice_relation"] = "Facture > mensualité"
invoice_cross = pd.crosstab(invoice_view.invoice_relation, invoice_view.churn).reindex(columns=[0, 1], fill_value=0)
invoice_cross.columns = ["Non-churn", "Churn"]
display(invoice_cross)
display(pd.crosstab(invoice_view.retention_offer_declined.fillna("Non renseigné"), invoice_view.churn, dropna=False))
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
for label, color in zip([0, 1], CLASS_COLORS):
    values = invoice_view.loc[invoice_view.churn.eq(label), "invoice_ratio"].dropna()
    ax.hist(values, bins=np.linspace(0, max(1.1, invoice_view.invoice_ratio.max()), 35), density=True, histtype="step", linewidth=2, color=color, label=f"churn={label} ; n={len(values)}")
ax.set(title="Facture / mensualité selon la cible — lignes évaluables", xlabel="Ratio facture / mensualité", ylabel="Densité par classe")
ax.legend()
show_figure(fig)
print("Offre refusée exclue du modèle 02 ; temporalité de la facture et de ses dérivées à confirmer.")

evaluable = invoice_view.loc[valid_invoice]
rule_prediction = evaluable.invoice_ratio.lt(1).astype(int)
if not evaluable.empty:
    agreement = rule_prediction.eq(evaluable.churn).mean()
    display(Markdown(f"**Constat calculé :** sur {len(evaluable):,} snapshots évaluables, la règle « facture inférieure à la mensualité » coïncide avec la cible dans {agreement:.2%} des cas. {int((~valid_invoice).sum()):,} snapshots ne sont pas évaluables. Une telle séparation mérite une vérification de la génération du dataset et de la temporalité de facturation avant d'interpréter les scores du modèle."))


# Deuxième partie — Les dix KPI de churn

Les cellules suivantes reprennent les dix questions métier du notebook initial, après compréhension de la structure. Les taux sont calculés sur les snapshots étiquetés de la copie EDA après la même déduplication que le 01. Les comptes de clients distincts sont affichés pour contextualiser les effectifs, sans supposer l'indépendance des lignes.

Les valeurs invalides des sources métier sont traitées comme manquantes dans cette copie exploratoire. Les quintiles sont descriptifs et ne sont pas des seuils à réutiliser sur le test pour sélectionner un modèle.

## KPI 1 — Prévalence globale du churn

**Question :** quelle part des snapshots étiquetés est positive ?  
**Pourquoi :** établir la baseline et apprécier le déséquilibre des classes.  
**À interpréter :** la correspondance avec des résiliations distinctes dépend de l’audit des trajectoires de la première partie ; les dénominateurs client et snapshot restent différents.

In [ ]:
total = len(labeled)
positives = int(labeled.churn.sum())
negatives = total - positives
display(pd.Series({"Snapshots étiquetés": total, "Clients distincts": labeled.customer_id.nunique(),
                   "Snapshots positifs": positives, "Prévalence (%)": global_rate}, name="valeur").to_frame())
dist = pd.DataFrame({"Statut": ["Non-churn", "Churn"], "Observations": [negatives, positives]})
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
bars = ax.bar(dist.Statut, dist.Observations, color=CLASS_COLORS)
ax.bar_label(bars, padding=3)
ax.set(title="KPI 1 — Distribution de la cible", ylabel="Snapshots étiquetés")
show_figure(fig)
display(dist)

## KPI 2 — Churn par contrat

**Question :** les types d'engagement sont-ils associés à des taux différents ?  
**Hypothèse :** le contrat mensuel pourrait présenter un churn plus élevé.  
**Vigilance :** vérifier la normalisation des catégories et les effectifs.

In [ ]:
stats = summary(eda, ["contract"]).sort_values("churn_rate", ascending=False)
rate_plot(stats, "contract", "KPI 2 — Churn par contrat", horizontal=True)

## KPI 3 — Churn par ancienneté

**Question :** le taux varie-t-il au cours du cycle de vie client ?  
**Hypothèse :** les premiers mois pourraient constituer une période plus risquée.  
**Vigilance :** ancienneté dérivée temporairement de deux dates ; vérifier la chronologie.

In [ ]:
temp = eda.copy()
order = ["0–3", "3–6", "6–12", "12–24", "24–48", "48+"]
temp["tenure_group"] = pd.cut(temp.tenure_months, bins=[0, 3, 6, 12, 24, 48, np.inf], labels=order, right=False)
print("Anciennetés non évaluables exclues :", int(temp.tenure_group.isna().sum()))
stats = summary(temp.dropna(subset=["tenure_group"]), ["tenure_group"])
rate_plot(stats, "tenure_group", "KPI 3 — Churn selon l'ancienneté (mois)")

## KPI 4 — Churn et appels au support

**Question :** les sollicitations du support sont-elles associées au churn ?  
**Hypothèse :** un nombre d'appels élevé pourrait révéler des frictions.  
**Vigilance :** les appels doivent précéder la fenêtre du label pour être prédictifs.

In [ ]:
temp = eda.copy()
order = ["0", "1", "2", "3+"]
temp["support_group"] = temp["support_calls_3m"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})
print("Valeurs non évaluables exclues :", int(temp["support_group"].isna().sum()))
stats = summary(temp.dropna(subset=["support_group"]), ["support_group"])
stats["support_group"] = pd.Categorical(stats["support_group"], categories=order, ordered=True)
stats = stats.sort_values("support_group")
rate_plot(stats, "support_group", "KPI 4 — Support et churn")

## KPI 5 — Churn et retards de paiement

**Question :** les incidents de paiement sont-ils associés au churn ?  
**Hypothèse :** plusieurs retards pourraient être un signal de risque.  
**Vigilance :** la période couverte par `late_payments` doit être confirmée.

In [ ]:
temp = eda.copy()
order = ["0", "1", "2", "3+"]
temp["payment_group"] = temp["late_payments"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})
print("Valeurs non évaluables exclues :", int(temp["payment_group"].isna().sum()))
stats = summary(temp.dropna(subset=["payment_group"]), ["payment_group"])
stats["payment_group"] = pd.Categorical(stats["payment_group"], categories=order, ordered=True)
stats = stats.sort_values("payment_group")
rate_plot(stats, "payment_group", "KPI 5 — Paiements et churn")

## KPI 6 — Churn selon le NPS

**Question :** la satisfaction déclarée est-elle liée au churn ?  
**Hypothèse :** les détracteurs pourraient résilier davantage.  
**Vigilance :** le NPS est souvent manquant ; conserver une catégorie « Non renseigné ». Les seuils supposent une échelle de 0 à 10.

In [ ]:
temp = eda.copy()
temp["nps_group"] = pd.cut(temp.nps, bins=[-.1, 6, 8, 10], labels=["Détracteurs (0–6)", "Passifs (>6–8)", "Promoteurs (>8–10)"])
temp["nps_group"] = temp.nps_group.astype("string").fillna("Non renseigné ou invalide")
stats = summary(temp, ["nps_group"]).sort_values("churn_rate", ascending=False)
rate_plot(stats, "nps_group", "KPI 6 — Satisfaction NPS et churn", horizontal=True)

## KPI 7 — Churn selon le prix mensuel

**Question :** le churn varie-t-il selon le tarif de l'abonnement ?  
**Hypothèse :** certaines tranches de prix pourraient être plus exposées.  
**Méthode :** quintiles de `monthly_fee` ; les bornes sont calculées sur ce dataset à des fins descriptives.

Ces quintiles sont uniquement exploratoires. La feature de modélisation `monthly_fee_band` du 01 utilise trois tranches apprises exclusivement sur le train, puis figées pour les autres périodes.

In [ ]:
temp = eda.copy()
if temp["monthly_fee"].nunique() >= 2:
    temp["fee_group"] = pd.qcut(temp["monthly_fee"], q=5, duplicates="drop")
    print("Valeurs non évaluables exclues :", int(temp["fee_group"].isna().sum()))
    stats = summary(temp.dropna(subset=["fee_group"]), ["fee_group"])
    rate_plot(stats, "fee_group", "KPI 7 — Prix mensuel et churn (€)")
else:
    print("Quintiles non calculables : moins de deux valeurs distinctes.")

### KPI 7 bis — Visualiser le découpage de `monthly_fee_band`

**Question :** à quoi ressemblent les trois tranches tarifaires proposées et quels profils de churn regroupent-elles ?

**Méthode :** reproduire les tertiles des mensualités valides des snapshots étiquetés de janvier–juillet 2024, exactement comme dans le 01. Les bornes ne sont pas arrondies pour l'affectation ; elles sont arrondies uniquement dans les légendes. Les seuils supérieurs appartiennent à la tranche de gauche. Les ex æquo peuvent rendre les groupes légèrement inégaux ou réduire leur nombre.

On affiche ensuite la distribution des prix du train, les volumes et les taux de churn par tranche sur train et validation (août–septembre), avec les mêmes bornes. « Peu coûteux » et « Coûteux » sont relatifs au train. La copie `fee_preview` sert uniquement à la visualisation : ni les données sources ni les CSV splittés ne sont modifiés. La création destinée au modèle reste dans le notebook 01.


In [ ]:
fee_preview = eda.loc[eda.churn.notna()].copy()
fee_preview_month = fee_preview.snapshot_date.dt.to_period("M")
fee_preview["split"] = np.select(
    [fee_preview_month.between("2024-01", "2024-07"),
     fee_preview_month.between("2024-08", "2024-09")],
    ["Train", "Validation"], default="Hors périmètre affiché",
)
fee_preview = fee_preview.loc[fee_preview.split.isin(["Train", "Validation"])].copy()
preview_train_fee = fee_preview.loc[fee_preview.split.eq("Train"), "monthly_fee"]
preview_train_fee = preview_train_fee.where(preview_train_fee.ge(0) & np.isfinite(preview_train_fee)).dropna()
if preview_train_fee.empty:
    raise ValueError("Aucune mensualité valide dans le train pour visualiser les tranches.")
preview_cuts = np.unique(preview_train_fee.quantile([1 / 3, 2 / 3]).to_numpy(dtype=float))
preview_cuts = preview_cuts[(preview_cuts > preview_train_fee.min()) & (preview_cuts < preview_train_fee.max())]
preview_edges = np.r_[-np.inf, preview_cuts, np.inf]
preview_labels = {
    1: ["Intermédiaire"], 2: ["Peu coûteux", "Coûteux"],
    3: ["Peu coûteux", "Intermédiaire", "Coûteux"],
}[len(preview_edges) - 1]
fee_preview["monthly_fee_band"] = pd.Series(pd.cut(
    fee_preview.monthly_fee.where(fee_preview.monthly_fee.ge(0) & np.isfinite(fee_preview.monthly_fee)),
    bins=preview_edges, labels=preview_labels, right=True,
), index=fee_preview.index).astype("string").fillna("Non renseigné")
preview_colors = dict(zip(preview_labels, ["#059669", "#2563EB", "#E11D48"] if len(preview_labels) == 3 else ["#059669", "#E11D48"]))
preview_colors["Non renseigné"] = "#94A3B8"
preview_order = preview_labels + ["Non renseigné"]
print("Bornes exactes du train (€) :", preview_edges)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.25))
visual_bins = np.linspace(preview_train_fee.min(), preview_train_fee.max(), 45)
for band in preview_labels:
    values = fee_preview.loc[fee_preview.split.eq("Train") & fee_preview.monthly_fee_band.eq(band), "monthly_fee"].dropna()
    ax.hist(values, bins=visual_bins, color=preview_colors[band], alpha=.75, label=f"{band} ; n={len(values):,}")
for cut in preview_cuts:
    ax.axvline(cut, color=COLORS["dark"], linestyle="--", linewidth=1.5)
    ax.text(cut, .97, f" {cut:.2f} €", transform=ax.get_xaxis_transform(), va="top", rotation=90)
ax.set(title="Découpage tarifaire — distribution des mensualités du train",
       xlabel="monthly_fee (€)", ylabel="Snapshots avec mensualité valide")
ax.legend(loc="upper right")
show_figure(fig)


#### Quelle taille ont les groupes ?

Les barres montrent les effectifs et la part des snapshots dans chaque split. Les mensualités absentes ou invalides restent visibles en gris. Les tranches sont définies sur les prix valides du train ; elles ne garantissent pas trois groupes d'égale taille sur validation. Les clients distincts figurent dans le tableau, mais peuvent contribuer à plusieurs tranches au fil du suivi.


In [ ]:
fee_preview_summary = summary(fee_preview, ["split", "monthly_fee_band"])
fee_preview_summary["share_pct"] = fee_preview_summary.observations / fee_preview_summary.groupby("split").observations.transform("sum") * 100
preview_baseline = fee_preview.groupby("split").churn.mean() * 100
fee_preview_summary["lift"] = fee_preview_summary.churn_rate / fee_preview_summary.split.map(preview_baseline)
display(fee_preview_summary)
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.7))
for ax, split_name in zip(axes, ["Train", "Validation"]):
    table = fee_preview_summary.loc[fee_preview_summary.split.eq(split_name)].set_index("monthly_fee_band").reindex(preview_order)
    table = table.loc[table.observations.notna()]
    bars = ax.bar(table.index, table.observations, color=[preview_colors[b] for b in table.index])
    labels = [f"{int(row.observations):,} ({row.share_pct:.1f} %)" for row in table.itertuples()]
    ax.bar_label(bars, labels=labels, padding=3)
    ax.set(title=f"Répartition des tranches — {split_name}", ylabel="Snapshots")
    ax.margins(y=.2)
show_figure(fig)


#### Les abonnements coûteux churnent-ils davantage ?

Comparer les taux de churn **et** les volumes. Chaque ligne pointillée indique la prévalence du split correspondant, pas celle de toute l'année. Le lift du tableau vaut `taux du groupe / taux global du split`. Un taux supérieur décrit une association ; il ne prouve pas que le prix provoque le churn. Aucun seuil n'est choisi à partir du test.


In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT * 1.2))
x_positions = np.arange(len(preview_order))
width = .36
for offset, split_name, color in [(-width / 2, "Train", "#2563EB"), (width / 2, "Validation", "#E11D48")]:
    table = fee_preview_summary.loc[fee_preview_summary.split.eq(split_name)].set_index("monthly_fee_band").reindex(preview_order)
    bars = ax.bar(x_positions + offset, table.churn_rate.to_numpy(), width, label=split_name, color=color)
    bar_labels = [f"{value:.2f} %" if pd.notna(value) else "" for value in table.churn_rate]
    ax.bar_label(bars, labels=bar_labels, padding=3, fontsize=9)
    if split_name in preview_baseline.index:
        ax.axhline(preview_baseline.loc[split_name], color=color, linestyle="--", alpha=.6,
                   label=f"Prévalence {split_name.lower()}")
ax.set_xticks(x_positions, preview_order)
ax.set(title="Churn selon la tranche tarifaire — mêmes bornes train / validation",
       xlabel="monthly_fee_band", ylabel="Churn par snapshot (%)")
ax.margins(y=.3)
ax.legend(ncol=2, fontsize=9)
show_figure(fig)


## KPI 8 — Churn selon la consommation de données

**Question :** l'utilisation du service est-elle liée au churn ?  
**Hypothèse :** une faible consommation pourrait être associée à un désengagement ; la relation peut aussi être non linéaire.  
**Méthode :** quintiles de `data_gb`.

In [ ]:
temp = eda.copy()
if temp["data_gb"].nunique() >= 2:
    temp["usage_group"] = pd.qcut(temp["data_gb"], q=5, duplicates="drop")
    print("Valeurs non évaluables exclues :", int(temp["usage_group"].isna().sum()))
    stats = summary(temp.dropna(subset=["usage_group"]), ["usage_group"])
    rate_plot(stats, "usage_group", "KPI 8 — Consommation data et churn (Go)")
else:
    print("Quintiles non calculables : moins de deux valeurs distinctes.")

## KPI 9 — Interaction contrat × ancienneté

**Question :** l'association ancienneté–churn dépend-elle du contrat ?  
**Hypothèse :** les nouveaux clients mensuels pourraient être particulièrement exposés.  
**Vigilance :** masquer les taux des groupes de moins de 100 observations ; afficher néanmoins leurs effectifs. Le seuil est exploratoire.

In [ ]:
temp = eda.copy()
order = ["0–3", "3–6", "6–12", "12–24", "24–48", "48+"]
temp["tenure_group"] = pd.cut(temp.tenure_months, bins=[0, 3, 6, 12, 24, 48, np.inf], labels=order, right=False)
stats = summary(temp.dropna(subset=["tenure_group", "contract"]), ["tenure_group", "contract"])
MIN_OBSERVATIONS = 100
stats["rate_display"] = stats.churn_rate.where(stats.observations.ge(MIN_OBSERVATIONS))
heatmap = stats.pivot(index="tenure_group", columns="contract", values="rate_display").reindex(order)
counts = stats.pivot(index="tenure_group", columns="contract", values="observations").reindex(index=heatmap.index, columns=heatmap.columns).fillna(0)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(PANEL_HEIGHT, len(order) * .65)))
masked = np.ma.masked_invalid(heatmap.to_numpy(dtype=float))
im = ax.imshow(masked, cmap="Reds", aspect="auto", vmin=0)
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns)
ax.set_yticks(range(len(order)), order)
for i in range(len(order)):
    for j in range(len(heatmap.columns)):
        value, count = heatmap.iloc[i, j], int(counts.iloc[i, j])
        label = f"{value:.2f}%\nn={count}" if pd.notna(value) else f"n={count}\npetit groupe"
        ax.text(j, i, label, ha="center", va="center", fontsize=9)
ax.set(title="KPI 9 — Contrat × ancienneté", xlabel="Contrat", ylabel="Ancienneté (mois)")
fig.colorbar(im, ax=ax, label="Churn (%) ; groupes ≥ 100 snapshots")
show_figure(fig)
display(stats)

## KPI 10 — Stabilité temporelle du churn

**Question :** la prévalence change-t-elle selon les mois de snapshot ?  
**Hypothèse :** des variations peuvent refléter un changement du risque ou de la composition du panel.  
**Vigilance :** ce n'est pas automatiquement le taux de résiliation *survenue* durant le mois.

In [ ]:
temp = eda.assign(month=eda.snapshot_date.dt.to_period("M").astype(str))
stats = summary(temp, ["month"]).sort_values("month")
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
ax.bar(stats.month, stats.observations, color="#CBD5E1", label="Snapshots étiquetés")
ax.set(xlabel="Mois du snapshot", ylabel="Snapshots", title="KPI 10 — Volumes et prévalence mensuels")
ax.tick_params(axis="x", labelrotation=35)
rate_ax = ax.twinx()
rate_ax.plot(stats.month, stats.churn_rate, "o-", color=COLORS["red"], label="Taux de churn")
rate_ax.axhline(global_rate, linestyle="--", color=COLORS["blue"], label="Prévalence globale")
rate_ax.set_ylabel("Churn par snapshot (%)")
rate_ax.grid(False)
rate_ax.legend(loc="upper right")
show_figure(fig)
display(stats)

## Synthèse — Ce que la structure du fichier change dans l'analyse

Les constats ci-dessous sont calculés à partir des données, et non préremplis. Ils décrivent la période disponible ; ils ne constituent pas une validation du modèle.

In [ ]:
structure_findings = pd.DataFrame([
    {"question": "Quelle unité ?", "constat": f"{len(df):,} snapshots retenus pour {df.customer_id.nunique():,} clients", "conséquence": "Analyser séparément lignes et clients ; ne pas supposer leur indépendance."},
    {"question": "Clients répétés ?", "constat": f"{client_counts.gt(1).sum():,} clients vus plusieurs fois", "conséquence": "La longueur du suivi pondère les distributions par snapshot."},
    {"question": "Cible par ligne ou client ?", "constat": f"{global_rate:.2f}% de snapshots positifs ; {client_labels.positive_snapshots.gt(0).mean():.2%} des clients avec un churn observé", "conséquence": "Ces deux proportions ont des dénominateurs et horizons différents."},
    {"question": "Chronologie fiable ?", "constat": f"{len(post_churn):,} lignes après churn observé", "conséquence": "Confronter à la règle de sortie du panel ; absence du fichier ≠ preuve de churn."},
    {"question": "Facturation suspecte ?", "constat": "Voir le croisement facture / cible de la section 14", "conséquence": "Confirmer la disponibilité temporelle ; comparer les ablations sur validation dans le 01."},
])
display(structure_findings)

## Synthèse EDA — Hypothèses pour la prochaine phase

Après exécution, renseigner ce tableau **à partir des résultats réels**, sans inférer de causalité.

| KPI | Constat chiffré | Hypothèse à tester | Limite | Piste future (non implémentée) |
|---|---|---|---|---|
| 1. Prévalence | À compléter | Déséquilibre éventuel | Snapshots répétés | Métriques adaptées |
| 2. Contrat | À compléter | Engagement lié au churn | Composition des groupes | Encodage du contrat |
| 3. Ancienneté | À compléter | Risque non linéaire | Effet de sélection | Transformation de l'ancienneté |
| 4. Support | À compléter | Friction client | Chronologie des appels | Indicateurs support |
| 5. Paiements | À compléter | Incidents liés au churn | Fenêtre inconnue | Indicateurs de paiement |
| 6. NPS | À compléter | Satisfaction liée au churn | Non-réponse | NPS et disponibilité |
| 7. Prix | À compléter | Sensibilité tarifaire | Facteurs de confusion | Transformations du prix |
| 8. Usage | À compléter | Désengagement / non-linéarité | Valeurs extrêmes | Transformations d'usage |
| 9. Interaction | À compléter | Effet conjoint | Petits groupes | Interaction contrat × ancienneté |
| 10. Temps | À compléter | Instabilité des segments | Composition du panel | Validation temporelle |

### Questions métier à clarifier

1. Le label est défini à 30 jours par l’énoncé ; à quel instant exact devient-il disponible ?
2. L’audit observe-t-il des churns multiples ou des observations après résiliation ? Comment expliquer ces cas s’il y en a ?
3. À quelle date chaque variable est-elle connue ?
4. Les taux doivent-ils être analysés par snapshot ou par client pour la décision métier ?

**Fin de l'EDA :** aucune feature définitive, aucun entraînement de modèle. Le notebook 01 reprend la lecture et l’audit pour le feature engineering ; le 02 entraîne les modèles et le 03 approfondit l’évaluation et les SHAP. L’exploration de toute l’année signifie que le test préparé dans le 01 n’est pas aveugle.